# 04 — Hardware Alignment

Demonstrate the hardware simulator interface:
1. Connect to the simulator device
2. Read frames and process through the pipeline
3. Run inference and measure latency
4. Check timestamp consistency and frame drops

In [ ]:
import sys, pathlib, time
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from radar_drone_vision.datasets.synthetic import SyntheticGenerator
from radar_drone_vision.signal.framing import frame_signal
from radar_drone_vision.signal.complex_log_fft import regularized_complex_log_fft
from radar_drone_vision.classical.sra import SubspaceReliabilityAnalysis

## 1. Prepare a trained model for inference

In [ ]:
FRAME_SIZE = 64
HOP_SIZE = 32
N_FFT = 64

gen = SyntheticGenerator(seed=42, sample_duration_s=0.25, sample_rate_hz=2000.0)
train_samples = gen.generate_balanced_dataset(n_per_class=100)

X_train, y_train = [], []
for s in train_samples:
    frames = frame_signal(s.signal, frame_size=FRAME_SIZE, hop_size=HOP_SIZE)
    feat = regularized_complex_log_fft(frames, n_fft=N_FFT, feature_mode="magnitude_only")
    X_train.append(feat.ravel())
    y_train.append(s.label_binary)

X_train = np.array(X_train)
y_train = np.array(y_train)

sra = SubspaceReliabilityAnalysis(m_uav=5, m_non_uav=15, ridge=1e-4)
sra.fit(X_train, y_train)
print("Model fitted for inference demo.")

## 2. Simulate live frames

In [ ]:
# Generate test signals to simulate a live stream
test_gen = SyntheticGenerator(seed=99, sample_duration_s=0.25, sample_rate_hz=2000.0)
test_samples = test_gen.generate_balanced_dataset(n_per_class=20)

latencies = []
predictions = []
true_labels = []

for sample in test_samples:
    t_start = time.perf_counter()

    frames = frame_signal(sample.signal, frame_size=FRAME_SIZE, hop_size=HOP_SIZE)
    feat = regularized_complex_log_fft(frames, n_fft=N_FFT, feature_mode="magnitude_only")
    pred = sra.predict(feat.ravel().reshape(1, -1), threshold=1.0)

    t_end = time.perf_counter()
    latencies.append((t_end - t_start) * 1000)  # ms
    predictions.append(pred[0])
    true_labels.append(sample.label_binary)

latencies = np.array(latencies)
print(f"Processed {len(latencies)} frames")
print(f"Latency — mean: {latencies.mean():.2f} ms, "
      f"median: {np.median(latencies):.2f} ms, "
      f"p99: {np.percentile(latencies, 99):.2f} ms")

## 3. Latency distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(latencies, bins=20, color="steelblue", edgecolor="white")
ax.axvline(latencies.mean(), color="red", linestyle="--", label=f"Mean={latencies.mean():.2f}ms")
ax.set_xlabel("Inference latency (ms)")
ax.set_ylabel("Count")
ax.set_title("Per-sample inference latency")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Prediction accuracy on simulated stream

In [ ]:
from radar_drone_vision.eval.metrics import compute_all_metrics

metrics = compute_all_metrics(np.array(true_labels), np.array(predictions))
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"F1:       {metrics['f1']:.4f}")
print()
print(metrics['classification_report'])